In [ ]:
#imports
import numpy as np
from astropy.visualization import ZScaleInterval, LinearStretch, ImageNormalize
from astropy.io import fits
from photutils.background import Background2D, MedianBackground, SExtractorBackground, BkgIDWInterpolator
from astropy.stats import SigmaClip
import matplotlib.pyplot as plt
from photutils.segmentation import detect_threshold, detect_sources
from astropy.convolution import convolve, Gaussian2DKernel
from scipy.ndimage import binary_dilation

In [ ]:
# Open the FITS file
hdul = fits.open('ERO_nonDGL_subregion.fits')

# View the header information (metadata)
print(hdul[0].header)

# Extract the image data into a standard NumPy array
image_data = hdul[0].data
zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(image_data)

In [ ]:
# Visualize the array
fig, ax = plt.subplots()
im = ax.imshow(image_data, vmin=vmin, vmax=vmax, origin='upper')
ax.set_title("Z-Scaled FITS Image")
plt.colorbar(im, ax=ax)
plt.figure(figsize=(50, 50))
plt.show()

In [ ]:
nan_mask = ~np.isfinite(image_data)
kernel = Gaussian2DKernel(x_stddev=2.0)

def source_mask(data, n_sigma, dilate_iter):
    d = np.where(nan_mask, 0.0, data)
    thr = detect_threshold(d, n_sigma=n_sigma, mask=nan_mask)
    segm = detect_sources(convolve(d, kernel), thr, n_pixels=5)
    if segm is None:
        return nan_mask.copy()
    m = binary_dilation(segm.data > 0, iterations=dilate_iter)
    return m | nan_mask

def fit_bkg(data, mask, box=48, filt=3):
    return Background2D(
        data, box_size=(box, box), filter_size=(filt, filt),
        sigma_clip=SigmaClip(sigma=3.0, maxiters=10),
        bkg_estimator=MedianBackground(),
        mask=mask, exclude_percentile=40)

# pass 1: rough mask from raw data
mask1 = source_mask(image_data, n_sigma=3.0, dilate_iter=6)
bkg1 = fit_bkg(image_data, mask1)
clean1 = image_data - bkg1.background

# pass 2: better mask from the flattened image, lower threshold
mask2 = source_mask(clean1, n_sigma=1.5, dilate_iter=8)
bkg2 = fit_bkg(image_data, mask2)
clean_data = image_data - bkg2.background

norm = ImageNormalize(clean_data, interval=ZScaleInterval(), stretch=LinearStretch())

# Visualize the Before and After
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)
ax1.imshow(image_data, origin='upper', cmap='gray', vmin=vmin, vmax=vmax) #0, 1000
ax1.set_title('Original Data (with sky glow/noise)')

ax2.imshow(clean_data, origin='upper', cmap='gray', norm=norm)
ax2.set_title('Cleaned Data (background subtracted)')
plt.show()

In [ ]:
norm = ImageNormalize(clean1, interval=ZScaleInterval(), stretch=LinearStretch())

def overlay(ax, img, mask, title, color='red', alpha=0.45):
    ax.imshow(img, origin='lower', cmap='gray', norm=norm)
    ov = np.ma.masked_where(~mask, np.ones(mask.shape))
    ax.imshow(ov, origin='lower', cmap=plt.matplotlib.colors.ListedColormap([color]),
              alpha=alpha, vmin=0, vmax=1)
    ax.set_title(f"{title}  ({mask.mean()*100:.1f}% masked)")

# full field: pass 1 vs pass 2
fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)
overlay(axes[0], clean1, mask1, 'Pass 1 mask (3σ, dilate 6)')
overlay(axes[1], clean1, mask2, 'Pass 2 mask (1.5σ, dilate 8)', color='white')
plt.tight_layout(); 
plt.show()